In [1]:
!pip install langchain langchain-community langchain-core pypdf -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from google.colab import files
uploaded = files.upload()

Saving Recruitment Policy.txt to Recruitment Policy.txt
Saving 4. Attendance Policy.txt to 4. Attendance Policy.txt
Saving company policy.txt to company policy.txt
Saving Employee Benefits Policy.txt to Employee Benefits Policy.txt
Saving HR-Policy.pdf to HR-Policy.pdf
Saving Leave Policy.txt to Leave Policy.txt
Saving Payroll Policy.txt to Payroll Policy.txt


In [3]:
from pypdf import PdfReader
documents = {}

for filename in uploaded.keys():
    text = ""
    if filename.lower().endswith(".pdf"):
        reader = PdfReader(filename)
        for page in reader.pages:
            text += page.extract_text() or ""

    elif filename.lower().endswith(".txt"):
        with open(filename, "r", encoding="utf-8") as file:
            text = file.read()

    else:
        print("Unsupported file:", filename)
        continue

    documents[filename] = text
    print("Extracted:", filename)

print("\nTotal documents:", len(documents))

Extracted: Recruitment Policy.txt
Extracted: 4. Attendance Policy.txt
Extracted: company policy.txt
Extracted: Employee Benefits Policy.txt
Extracted: HR-Policy.pdf
Extracted: Leave Policy.txt
Extracted: Payroll Policy.txt

Total documents: 7


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)


In [5]:
from langchain_core.documents import Document
langchain_docs = []
for filename, text in documents.items():
    doc = Document(
        page_content=text,
        metadata={"source": filename}
    )
    langchain_docs.append(doc)

print("Total documents:", len(langchain_docs))

chunks = splitter.split_documents(langchain_docs)

print("Total chunks:", len(chunks))

Total documents: 7
Total chunks: 51


In [6]:
print(chunks[0].page_content)
print(chunks[0].metadata)

Recruitment Policy
Introduction
A Recruitment Policy is a set of guidelines that explains how an organization identifies, attracts, evaluates, and selects suitable candidates for job vacancies. It ensures that the hiring process is fair, transparent, and based on candidates' qualifications and skills.


Objectives
- To hire qualified and skilled employees.
- To ensure a fair and unbiased selection process.
- To fill vacant positions within the required time.
- To maintain consistency in recruitment procedures.
- To attract talented candidates who match organizational requirements.
{'source': 'Recruitment Policy.txt'}


In [7]:
!pip install -q langchain-chroma  langchain-huggingface

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_function,
    collection_name="hr_knowledge"
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
!pip install -q rank-bm25

In [10]:
from rank_bm25 import BM25Okapi
tokenized_chunks = [
    chunk.page_content.lower().split()
    for chunk in chunks
]
bm25 = BM25Okapi(tokenized_chunks)

In [11]:
query = "what is the leave policy"
tokenized_query = query.lower().split()

results = bm25.get_top_n(
    tokenized_query,
    chunks,
    n=3
)

for i, doc in enumerate(results):
    print(f"\nResult {i+1}:")
    print(doc.page_content[:500])
    print("Source:", doc.metadata["source"])


Result 1:
confidential support mechanisms. 
  
 
PERFORMANCE MANAGEMENT AND APPRAISAL 
  
 The performance management policy is one of the HR policies created to give employees a 
clear understanding of what is expected of them. It provides an appropriate framework 
within which the performance of an employee can be managed. Also, this can be done 
through the implementation of various tools, techniques, as well as processes. 
  
 While performance management includes identifying, measuring, managing, and
Source: HR-Policy.pdf

Result 2:
Therefore, it may be tempting to only focus on the competitive market rates, however, it is 
important to pay heed to the compliances required for every employee’s wages. 
At Rikalp we follow all the statutory norms including employee provident fund, employee 
state insurance of the eligible employees. 
  
LEAVE POLICY 
  
The leave policy is created to give employees a clear picture of the number of leaves an 
employee is allowed to take annually. Fu

In [12]:
query = "What is the leave policy?"

vector_results = vectorstore.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(vector_results):
    print(f"\nResult {i+1}:")
    print(doc.page_content[:500])
    print("Source:", doc.metadata["source"])


Result 1:
1. Leave Management Policy
Introduction
A Leave Management Policy is a set of rules and guidelines that explains how employees can request leave from work. It helps organizations manage employee absences while ensuring that daily business operations continue smoothly.


Objectives
- To provide employees with sufficient time off for personal needs, health issues, and vacations.
- To maintain a proper record of employee leaves.
- To ensure fair and consistent leave approval procedures.
- To manage
Source: Leave Policy.txt

Result 2:
a) Female Employee with over one-year continuous service shall be entitled to 26 weeks 
maternity leave subject to production of medical certificate being their pregnant and 
the probable date of confinement. Not with standing anything the same female 
employee may be allowed to combine any other kind of leave subject to availability 
of leave title along with the maternity leave provided that the total leave so availed does not exceed 60 days. Mat

In [13]:
def reciprocal_rank_fusion(bm25_results, vector_results, k=60):

    scores = {}
    documents = {}

    for rank, doc in enumerate(bm25_results):
        doc_id = doc.metadata["source"] + doc.page_content
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
        documents[doc_id] = doc

    for rank, doc in enumerate(vector_results):
        doc_id = doc.metadata["source"] + doc.page_content
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
        documents[doc_id] = doc

    ranked_ids = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return [documents[doc_id] for doc_id in ranked_ids]

In [14]:
hybrid_results = reciprocal_rank_fusion(
    bm25_results=results,
    vector_results=vector_results
)

for i, doc in enumerate(hybrid_results[:3]):
    print(f"\nResult {i+1}:")
    print(doc.page_content[:500])
    print("Source:", doc.metadata["source"])


Result 1:
Therefore, it may be tempting to only focus on the competitive market rates, however, it is 
important to pay heed to the compliances required for every employee’s wages. 
At Rikalp we follow all the statutory norms including employee provident fund, employee 
state insurance of the eligible employees. 
  
LEAVE POLICY 
  
The leave policy is created to give employees a clear picture of the number of leaves an 
employee is allowed to take annually. Furthermore, it should also mention public holi
Source: HR-Policy.pdf

Result 2:
a) Female Employee with over one-year continuous service shall be entitled to 26 weeks 
maternity leave subject to production of medical certificate being their pregnant and 
the probable date of confinement. Not with standing anything the same female 
employee may be allowed to combine any other kind of leave subject to availability 
of leave title along with the maternity leave provided that the total leave so availed does not exceed 60 days. Matern

In [15]:
!pip install sentence-transformers

In [16]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [17]:
query = "What is the leave policy?"

# Prepare query-document pairs
pairs = [
    [query, doc.page_content]
    for doc in hybrid_results
]

# Calculate relevance scores
scores = reranker.predict(pairs)

# Sort documents by relevance score
reranked_results = sorted(
    zip(scores, hybrid_results),
    key=lambda x: x[0],
    reverse=True
)

# Display top 3 results
for i, (score, doc) in enumerate(reranked_results[:3]):
    print(f"\nResult {i+1}")
    print("Relevance Score:", round(float(score), 4))
    print("Content:", doc.page_content[:500])
    print("Source:", doc.metadata["source"])


Result 1
Relevance Score: 8.6068
Content: 1. Leave Management Policy
Introduction
A Leave Management Policy is a set of rules and guidelines that explains how employees can request leave from work. It helps organizations manage employee absences while ensuring that daily business operations continue smoothly.


Objectives
- To provide employees with sufficient time off for personal needs, health issues, and vacations.
- To maintain a proper record of employee leaves.
- To ensure fair and consistent leave approval procedures.
- To manage
Source: Leave Policy.txt

Result 2
Relevance Score: 6.0442
Content: Therefore, it may be tempting to only focus on the competitive market rates, however, it is 
important to pay heed to the compliances required for every employee’s wages. 
At Rikalp we follow all the statutory norms including employee provident fund, employee 
state insurance of the eligible employees. 
  
LEAVE POLICY 
  
The leave policy is created to give employees a clear picture o

In [18]:
!pip install -q --upgrade langchain-core langchain-groq

In [19]:
!pip install -q --force-reinstall --no-deps langchain-groq==1.1.3

In [20]:
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("groq_api"),
    temperature=0
)


In [21]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an HR Knowledge Assistant.

Answer using only the provided HR context.

Rules:
1. Do not make up information.
2. If information is unavailable, say:
   "I could not find this information in the uploaded HR documents."
3. Mention the source document filenames.

HR Context:
{context}

Employee Question:
{question}

Provide the answer with source filenames.
""")

In [22]:
context = "\n\n".join(
    f"Source: {doc.metadata['source']}\n"
    f"Content: {doc.page_content}"
    for score, doc in reranked_results[:3]
)

print(context)

Source: Leave Policy.txt
Content: 1. Leave Management Policy
Introduction
A Leave Management Policy is a set of rules and guidelines that explains how employees can request leave from work. It helps organizations manage employee absences while ensuring that daily business operations continue smoothly.


Objectives
- To provide employees with sufficient time off for personal needs, health issues, and vacations.
- To maintain a proper record of employee leaves.
- To ensure fair and consistent leave approval procedures.
- To manage employee availability and workload effectively.

Source: HR-Policy.pdf
Content: Therefore, it may be tempting to only focus on the competitive market rates, however, it is 
important to pay heed to the compliances required for every employee’s wages. 
At Rikalp we follow all the statutory norms including employee provident fund, employee 
state insurance of the eligible employees. 
  
LEAVE POLICY 
  
The leave policy is created to give employees a clear pictur

In [23]:
messages = prompt.format_messages(
    context=context,
    question="What is the leave policy?"
)

response = llm.invoke(messages)

print(response.content)

**Leave Policy (as described in the uploaded documents)**  

- **Purpose & Objectives**  
  - Provides employees with sufficient time off for personal needs, health issues, and vacations.  
  - Maintains accurate records of employee leaves.  
  - Ensures fair and consistent approval procedures.  
  - Manages employee availability and workload effectively.  
  *(Source: Leave Policy.txt)*  

- **Annual Leave Entitlements**  
  - All regular employees are entitled to **18 earned leaves** per calendar year.  
  - Employees who join mid‑year receive earned leaves on a **pro‑rata basis**.  
  *(Source: HR‑Policy.pdf)*  

- **Maternity Leave**  
  - Female employees with over one year of continuous service are entitled to **26 weeks of maternity leave** (full pay plus any applicable allowances).  
  - They may combine other leave types, provided the total leave taken does not exceed **60 days**.  
  - Employees covered under ESIC receive maternity leave through ESIC.  
  *(Source: HR‑Policy.

In [24]:
chat_history = []
def ask_hr_assistant(question):

    global chat_history

    # BM25 keyword retrieval
    tokenized_query = question.lower().split()

    bm25_results = bm25.get_top_n(
        tokenized_query,
        chunks,
        n=5
    )

    # Vector retrieval
    vector_results = vectorstore.similarity_search(
        question,
        k=5
    )

    # Hybrid retrieval
    hybrid_results = reciprocal_rank_fusion(
        bm25_results,
        vector_results
    )

    # Cross-Encoder reranking
    pairs = [
        [question, doc.page_content]
        for doc in hybrid_results
    ]

    scores = reranker.predict(pairs)

    reranked_results = sorted(
        zip(scores, hybrid_results),
        key=lambda x: x[0],
        reverse=True
    )

    # Prepare context with sources
    context = "\n\n".join(
        f"Source: {doc.metadata['source']}\n"
        f"Content: {doc.page_content}"
        for score, doc in reranked_results[:3]
    )

    # Include conversation history
    history = "\n".join(chat_history)

    messages = prompt.format_messages(
        context=context,
        question=f"""
Previous Conversation:
{history}

Current Question:
{question}
"""
    )

    response = llm.invoke(messages)

    chat_history.append(f"Employee: {question}")
    chat_history.append(f"Assistant: {response.content}")

    return response.content

In [25]:
print(ask_hr_assistant("What is the leave policy?"))

**Leave Policy (as outlined in the company documents)**  

- **Purpose & Objectives**  
  - Provide employees with sufficient time off for personal needs, health issues, and vacations.  
  - Maintain accurate records of employee leaves.  
  - Ensure fair and consistent approval procedures.  
  - Manage employee availability and workload effectively.  

- **Types of Leave**  
  1. **Casual Leave (CL)** – for personal work, emergencies, or unexpected situations.  
  2. **Sick Leave (SL)** – when an employee is unwell or needs medical attention.  
  3. **Earned Leave (EL)** – accumulated based on days worked; all regular employees are entitled to **18 ELs per calendar year** (pro‑rated for mid‑year joiners).  
  4. **Maternity Leave** – for eligible female employees during pregnancy and childbirth.  
  5. **Paternity Leave** – for eligible fathers after the birth of a child, per company policy.  
  6. **Unpaid Leave (LWP)** – taken when no paid leave balance is available or the employee i

In [26]:
print(ask_hr_assistant("What about sick leave?"))

**Sick Leave (SL)**  
- **Purpose:** Granted when an employee is unwell or needs medical attention.  
- **Application:** Employees should inform their manager as early as possible and submit the request through the HR portal or directly to the reporting manager.  
- **Paid Status:** Sick leave is a form of paid leave, as stated under “Paid Leave” in the Employee Benefits Policy.  

**Sources**  
- *Leave Policy.txt*  
- *Employee Benefits Policy.txt*


In [27]:
!pip install langgraph

In [28]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class HRState(TypedDict):
    question: str
    documents: list
    context: str
    answer: str

In [29]:
def retrieval_node(state: HRState):

    # Get question from workflow state
    question = state["question"]

    # BM25 keyword retrieval
    tokenized_query = question.lower().split()

    bm25_results = bm25.get_top_n(
        tokenized_query,
        chunks,
        n=5
    )

    # Vector retrieval
    vector_results = vectorstore.similarity_search(
        question,
        k=5
    )

    # Hybrid retrieval
    hybrid_results = reciprocal_rank_fusion(
        bm25_results,
        vector_results
    )

    return {
        "documents": hybrid_results
    }

print("Retrieval node updated successfully!")

Retrieval node updated successfully!


In [30]:
def reranking_node(state: HRState):
    question = state["question"]
    documents = state["documents"]
    pairs = [
        [question, doc.page_content]
        for doc in documents
    ]

    scores = reranker.predict(pairs)

    reranked_results = sorted(
        zip(scores, documents),
        key=lambda x: x[0],
        reverse=True
    )

    top_documents = [
        doc for score, doc in reranked_results[:3]
    ]

    return {
        "documents": top_documents
    }



In [41]:
def generation_node(state: HRState):

    documents = state["documents"]

    context = "\n\n".join(
        f"Source: {doc.metadata['source']}\n"
        f"Content: {doc.page_content}"
        for doc in documents
    )

    chat_history = state.get("chat_history", [])

    history_text = "\n".join(chat_history)

    question_with_history = f"""
Previous Conversation:
{history_text}

Current Question:
{state["question"]}
"""

    messages = prompt.format_messages(
        context=context,
        question=question_with_history
    )

    response = llm.invoke(messages)

    updated_history = chat_history + [
        f"Employee: {state['question']}",
        f"Assistant: {response.content}"
    ]

    return {
        "context": context,
        "answer": response.content,
        "chat_history": updated_history
    }

In [43]:
workflow = StateGraph(HRState)

workflow.add_node("retrieval", retrieval_node)
workflow.add_node("reranking", reranking_node)
workflow.add_node("generation", generation_node)

workflow.add_edge(START, "retrieval")
workflow.add_edge("retrieval", "reranking")
workflow.add_edge("reranking", "generation")
workflow.add_edge("generation", END)

hr_app = workflow.compile(
    checkpointer=memory
)

print("HR Assistant workflow compiled with conversational memory!")

HR Assistant workflow compiled with conversational memory!


In [34]:
result = hr_app.invoke({
    "question": "What is the leave policy?",
    "documents": [],
    "context": "",
    "answer": ""
})

print("Employee Question:")
print("What is the leave policy?")

print("\nHR Assistant Answer:")
print(result["answer"])

Employee Question:
What is the leave policy?

HR Assistant Answer:
**Leave Policy Overview**

The company’s leave policy outlines the types of leave available, the entitlement amounts, and the procedures for requesting and approving leave. Key points include:

| Leave Type | Description |
|------------|-------------|
| **Casual Leave (CL)** | For personal work, emergencies, or unexpected situations. |
| **Sick Leave (SL)** | Granted when an employee is unwell or needs medical attention. |
| **Earned Leave (EL)** | Accumulated based on days worked; regular employees receive 18 EL days per calendar year (pro‑rated for mid‑year joiners). |
| **Maternity Leave** | Provided to eligible female employees during pregnancy and childbirth. |
| **Paternity Leave** | Provided to eligible fathers after the birth of a child, per company policy. |
| **Unpaid Leave (LWP)** | Taken when no paid leave balance is available or the employee is not eligible for paid leave. |

**Key Policy Points**

- The po

In [36]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()
hr_app = workflow.compile(
    checkpointer=memory
)

In [37]:
config = {
    "configurable": {
        "thread_id": "hr_employee_1"
    }
}

In [38]:
result = hr_app.invoke(
    {
        "question": "What is the leave policy?",
        "documents": [],
        "context": "",
        "answer": ""
    },
    config=config
)

print(result["answer"])

**Leave Policy Overview**

The company’s leave policy outlines the types of leave available, the entitlement amounts, and the procedures for requesting and approving leave. Key points include:

| Leave Type | Description |
|------------|-------------|
| **Casual Leave (CL)** | For personal work, emergencies, or unexpected situations. |
| **Sick Leave (SL)** | Granted when an employee is unwell or needs medical attention. |
| **Earned Leave (EL)** | Accumulated based on days worked; regular employees receive 18 EL days per calendar year (pro‑rated for mid‑year joiners). |
| **Maternity Leave** | Provided to eligible female employees during pregnancy and childbirth. |
| **Paternity Leave** | Provided to eligible fathers after the birth of a child, per company policy. |
| **Unpaid Leave (LWP)** | Taken when no paid leave balance is available or the employee is not eligible for paid leave. |

**Key Policy Points**

- The policy ensures employees have sufficient time off for personal needs,

In [42]:
follow_up = hr_app.invoke(
    {
        "question": "How many earned leave days are provided per year?",
        "documents": [],
        "context": "",
        "answer": ""
    },
    config=config
)
print(follow_up["answer"])

The policy states that all regular employees are entitled to **18 Earned Leaves per calendar year**.  
*Source: HR‑Policy.pdf*


In [49]:
def ask_hr_assistant(question):

    result = hr_app.invoke(
        {
            "question": question,
            "documents": [],
            "context": "",
            "answer": ""
        },
        config=config
    )

    return result["answer"]


In [50]:
test_questions = [
    "What is the leave policy?",
    "What are the employee benefits?",
    "Explain the payroll policy.",
    "What is the attendance policy?",
    "What is the recruitment process?"
]

for question in test_questions:

    result = hr_app.invoke(
        {
            "question": question,
            "documents": [],
            "context": "",
            "answer": ""
        },
        config=config
    )

    print("\nEmployee Question:", question)
    print("HR Assistant Answer:", result["answer"])
    print("-" * 80)


Employee Question: What is the leave policy?
HR Assistant Answer: **Leave Policy (as outlined in the company documents)**  

- **Purpose & Objectives**  
  - Provide employees with sufficient time off for personal needs, health issues, and vacations.  
  - Maintain accurate records of employee leaves.  
  - Ensure fair and consistent approval procedures.  
  - Manage employee availability and workload effectively.  

- **Types of Leave**  
  1. **Casual Leave (CL)** – for personal work, emergencies, or unexpected situations.  
  2. **Sick Leave (SL)** – when an employee is unwell or needs medical attention.  
  3. **Earned Leave (EL)** – accumulated based on days worked; all regular employees are entitled to **18 ELs per calendar year** (pro‑rated for mid‑year joiners).  
  4. **Maternity Leave** – for eligible female employees during pregnancy and childbirth.  
  5. **Paternity Leave** – for eligible fathers after the birth of a child, per company policy.  
  6. **Unpaid Leave (LWP)*

In [51]:
question = "What is the recipe for making pizza?"

result = hr_app.invoke(
    {
        "question": question,
        "documents": [],
        "context": "",
        "answer": ""
    },
    config=config
)

print("Employee Question:", question)
print("\nHR Assistant Answer:")
print(result["answer"])

Employee Question: What is the recipe for making pizza?

HR Assistant Answer:
I could not find this information in the uploaded HR documents. (Source: HR-Policy.pdf)
